In [ ]:
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [ ]:
def parse_bbox(b):
    lon1, lat1, lon2, lat2 = map(float, b.split(","))
    return lon1, lat1, lon2, lat2

bboxes = {
    "bbox5 (PURPLE)": parse_bbox("27.2,82.348,35,83.2"),
    "bbox6 (CYAN)": parse_bbox("27,82.951,35.52,83.8"),
    "bbox7 (YELLOW)": parse_bbox("33,82.95,40.715,83.903"),
    "bbox8 (MAGENTA)": parse_bbox("33.07,82.34,40.1,83.297"),
    "bbox9 (BROWN)": parse_bbox("33.2,81.641,39.585,82.6"),
    "bbox10 (GREY)": parse_bbox("27.765,81.753,35,82.6"),
    "bbox11 (BLACK)": parse_bbox("38.38,82.95,46,83.913"),
    "bbox12 (WHITE)": parse_bbox("38.65,82.33,45.61,83.293"),
    "bbox13 (LIGHT GREEN)": parse_bbox("38.38,81.728,45.44,82.6")
}

bbox_colors = {
    "bbox5 (PURPLE)": "purple",
    "bbox6 (CYAN)": "cyan",
    "bbox7 (YELLOW)": "yellow",
    "bbox8 (MAGENTA)": "magenta",
    "bbox9 (BROWN)": "brown",
    "bbox10 (GREY)": "grey",
    "bbox11 (BLACK)": "black",
    "bbox12 (WHITE)": "white",
    "bbox13 (LIGHT GREEN)": "tab:green"
}

def bbox_to_region_name(lon1, lat1, lon2, lat2):
    fmt = lambda x: str(x).replace(".", "_")
    return f"region-{fmt(lon1)}-{fmt(lat1)}-{fmt(lon2)}-{fmt(lat2)}"

def get_region_for_point(lon, lat, bboxes):
    for name, (lon1, lat1, lon2, lat2) in bboxes.items():
        if lon1 <= lon <= lon2 and lat1 <= lat <= lat2:
            return bbox_to_region_name(lon1, lat1, lon2, lat2)
    return None



In [ ]:
file = 'IABP_buoys.csv'
df = pd.read_csv(file, header=0)

units = pd.read_csv(file, nrows=1, skiprows=1, header=None).iloc[0]
df = df.iloc[1:].reset_index(drop=True)

print("Units:")
print(units)

In [ ]:
df.head(25)

In [ ]:
# Ensure numeric
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# Add region column
df["region"] = df.apply(
    lambda row: get_region_for_point(row["longitude"], row["latitude"], bboxes),
    axis=1
)

# Filter to rows inside any region
df_filtered = df[df["region"].notna()].copy()

print("Original size:", len(df))
print("Filtered size:", len(df_filtered))
print("Unique regions found:", df_filtered["region"].unique())


# MAKE buoy in region CSV

In [ ]:
# Convert time to date only
df_filtered["date"] = pd.to_datetime(df_filtered["time"]).dt.date

# Group by region + date
region_daily = (
    df_filtered
    .groupby(["region", "date"])
    .agg(
        buoy_ids=("buoy_id", lambda x: ",".join(sorted(set(x.astype(str))))),
        n_buoys   =("buoy_id", lambda x: len(set(x)))
    )
    .reset_index()
)

region_daily.to_csv("region_daily_presence.csv", index=False)


# MAKE .tiff path buoy date pairs CSV

In [ ]:
region_daily = pd.read_csv("region_daily_presence.csv", parse_dates=["date"])
region_daily["date"] = region_daily["date"].dt.date  # ensure plain date

import glob

BASE = os.environ.get("SAR_DATASET_ROOT", "/path/to/SAR_sea_ice_dataset")  # set SAR_DATASET_ROOT env var

def find_tiffs_for_region_date(region_name, date):
    """
    region_name: e.g. 'region-27_0-82_951-35_52-83_8'
    date: python date object
    """
    year  = f"{date.year:04d}"
    month = f"{date.month:02d}"
    day   = f"{date.day:02d}"

    region_path = os.path.join(BASE, region_name, year, month, day)

    # Find all TIFF files under that directory
    tiffs = glob.glob(os.path.join(region_path, "*.tiff"))

    return tiffs

rows = []

for _, row in region_daily.iterrows():
    region_name = row["region"]
    date        = row["date"]
    buoy_ids    = row["buoy_ids"]
    n_buoys     = row["n_buoys"]

    tiff_files = find_tiffs_for_region_date(region_name, date)

    # If no TIFF files, skip this entry
    if len(tiff_files) == 0:
        continue

    # If multiple TIFFs exist, create one row per TIFF
    for tiff in tiff_files:
        rows.append({
            "tiff_path": tiff,
            "region": region_name,
            "date": str(date),
            "buoy_ids": buoy_ids,
            "n_buoys": n_buoys
        })

output = pd.DataFrame(rows)
output.to_csv("region_date_tiff_and_buoys.csv", index=False)

print("Wrote", len(output), "rows to region_date_tiff_and_buoys.csv")



# MAKE exact .tiff buoy time match CSV

In [ ]:
# Ensure time is datetime and lat/lon numeric
df_filtered = df_filtered.copy()
df_filtered["time"] = pd.to_datetime(df_filtered["time"])
df_filtered["latitude"] = pd.to_numeric(df_filtered["latitude"], errors="coerce")
df_filtered["longitude"] = pd.to_numeric(df_filtered["longitude"], errors="coerce")
df_filtered["time"] = pd.to_datetime(df_filtered["time"], utc=True).dt.tz_convert(None)


tiff_df = pd.read_csv("region_date_tiff_and_buoys.csv")

# Ensure date is a proper date
tiff_df["date"] = pd.to_datetime(tiff_df["date"]).dt.date

def tiff_time_from_path(tiff_path):
    fname = os.path.basename(tiff_path)
    stem = os.path.splitext(fname)[0]   # '20150204T0637'
    t = pd.to_datetime(stem, format="%Y%m%dT%H%M", utc=True)
    return t.tz_convert(None)  # make timezone-naive


rows = []
MAX_TIME_DIFF = 2 * 3600  # 1 hour in seconds

for _, row in tiff_df.iterrows():
    tiff_path = row["tiff_path"]
    region    = row["region"]
    date      = row["date"]

    sar_time = tiff_time_from_path(tiff_path)

    # Buoy obs in same region + same date
    candidates = df_filtered[
        (df_filtered["region"] == region) &
        (df_filtered["time"].dt.date == date)
    ].copy()

    if candidates.empty:
        continue

    # Time difference to SAR timestamp
    candidates["time_diff"] = (candidates["time"] - sar_time).abs()

    # For each buoy in region-date, find closest observation
    for buoy_id, sub in candidates.groupby("buoy_id"):
        idx = sub["time_diff"].idxmin()
        obs = sub.loc[idx]

        # keep only if diff <= 2 hours
        if obs["time_diff"].total_seconds() <= MAX_TIME_DIFF:
            rows.append({
                "tiff_path": tiff_path,
                "region": region,
                "buoy_date_time": obs["time"],
                "buoy_id": buoy_id,
                "buoy_type": obs["buoy_type"],
                "buoy_lat": obs["latitude"],
                "buoy_lon": obs["longitude"],
                "time_diff_seconds": obs["time_diff"].total_seconds()
            })


matched = pd.DataFrame(rows)

matched.to_csv("tiff_buoy_closest_observations.csv", index=False)
print("Wrote", len(matched), "rows to tiff_buoy_closest_observations.csv")


In [ ]:
df_matches = pd.read_csv("tiff_buoy_closest_observations.csv")

max_diff = df_matches["time_diff_seconds"].max()

print("Largest time difference (seconds):", max_diff)
print("Largest time difference (hours):", max_diff / 3600)

In [ ]:
unique_buoys = df_matches['buoy_id'].unique()
print(f"Unique buoys: {len(unique_buoys)}")

In [ ]:
unique_buoy_types = df_matches['buoy_type'].unique()
print(f"Unique buoy types: {unique_buoy_types}")

# 6h to 26h drift pairs

In [ ]:
df_matches["buoy_date_time"] = pd.to_datetime(df_matches["buoy_date_time"])
df_matches = df_matches.sort_values(["region", "buoy_id", "buoy_date_time"])

lower_s = 6 * 3600   # 6 hours
upper_s = 26 * 3600  # 26 hours

pairs = []
count_pairs = 0

for (region, buoy_id), sub in df_matches.groupby(["region", "buoy_id"]):
    times = sub["buoy_date_time"].values
    lats  = sub["buoy_lat"].values
    lons  = sub["buoy_lon"].values
    tiffs = sub["tiff_path"].values

    n = len(times)

    # Double loop: check ALL possible pairs
    for i in range(n):
        for j in range(i+1, n):

            dt_seconds = (times[j] - times[i]) / np.timedelta64(1, "s")

            # If dt is too large, break early (because times are sorted)
            if dt_seconds > upper_s:
                break

            # Only accept dt in valid drift window
            if dt_seconds >= lower_s:
                count_pairs += 1

                pairs.append({
                    "region": region,
                    "buoy_id": buoy_id,

                    # Drift start & end times
                    "t0": pd.Timestamp(times[i]),
                    "t1": pd.Timestamp(times[j]),
                    "dt_hours": dt_seconds / 3600,

                    # Start position
                    "t0_lat": lats[i],
                    "t0_lon": lons[i],

                    # End position
                    "t1_lat": lats[j],
                    "t1_lon": lons[j],

                    # Corresponding TIFF files
                    "tiff0_path": tiffs[i],
                    "tiff1_path": tiffs[j]
                })

print("Number of valid 6h–26h drift pairs:", count_pairs)

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv("valid_buoy_drift_pairs_dt6h_to_dt26h.csv", index=False)
pairs_df


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to date-only
pairs_df["date"] = pd.to_datetime(pairs_df["t0"]).dt.date

# Unique sorted dates
unique_dates = pd.to_datetime(sorted(pairs_df["date"].unique()))

plt.figure(figsize=(8, 2))

plt.plot(unique_dates, [1]*len(unique_dates), '|', markersize=15, color='#74c476')
plt.yticks([])
plt.title("Timeline of Available Buoy Data (Daily Coverage) in the SAR Domain")
plt.xlabel("Date")

plt.show()


In [ ]:
days_per_year = unique_dates.to_series().dt.year.value_counts().sort_index()
print(days_per_year)


In [ ]:
days_per_month = unique_dates.to_series().dt.month.value_counts().sort_index()
print(days_per_month)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Make sure pairs_df.t0 is datetime
pairs_df["t0"] = pd.to_datetime(pairs_df["t0"])

# Extract month of each drift pair's start day
months = pairs_df["t0"].dt.month

season_map = {
    "Winter (DJF)": [12, 1, 2],
    "Spring (MAM)": [3, 4, 5],
    "Summer (JJA)": [6, 7, 8],
    "Autumn (SON)": [9, 10, 11]
}

# Count how many pairs fall into each season
season_counts = pd.Series({
    season: months.isin(month_list).sum()
    for season, month_list in season_map.items()
})

print(season_counts)
print(f"Total drift pairs: {season_counts.sum()}")


In [ ]:
season_colors = {
    "Winter (DJF)": "#6baed6",   # icy blue
    "Spring (MAM)": "#74c476",   # green
    "Summer (JJA)": "#fdae6b",   # warm orange
    "Autumn (SON)": "#d95f0e"    # autumn brown-orange
}

colors = [season_colors[s] for s in season_counts.index]

plt.figure(figsize=(8,4))
ax = season_counts.plot(kind='bar', color=colors, edgecolor='black')

plt.title("IABP Drift Dataset: Valid 6-26h Pairs of SAR-Buoy Drift Pairs")
plt.ylabel("Number of Drift Pairs")
plt.xticks(rotation=0)
plt.tight_layout()

# Add numeric labels
for i, value in enumerate(season_counts):
    ax.text(
        i,
        value - max(season_counts) * 0.1,
        str(value),
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

plt.show()


# 24 +-2h drift pairs

In [ ]:
valid_pairs = pairs_df[
    (pairs_df["dt_hours"] >= 22) &
    (pairs_df["dt_hours"] <= 26)
].copy()

print("Valid 24±2h drift pairs:", len(valid_pairs))


In [ ]:
# Extract month of start time
valid_pairs["t0"] = pd.to_datetime(valid_pairs["t0"])
months = valid_pairs["t0"].dt.month

season_map = {
    "Winter (DJF)": [12, 1, 2],
    "Spring (MAM)": [3, 4, 5],
    "Summer (JJA)": [6, 7, 8],
    "Autumn (SON)": [9, 10, 11]
}

season_counts = pd.Series({
    season: months.isin(month_list).sum()
    for season, month_list in season_map.items()
})

print(season_counts)
print("Total valid 24±2h drift pairs:", season_counts.sum())


In [ ]:
import matplotlib.pyplot as plt

season_colors = {
    "Winter (DJF)": "#6baed6",   # icy blue
    "Spring (MAM)": "#74c476",   # green
    "Summer (JJA)": "#fdae6b",   # warm orange
    "Autumn (SON)": "#d95f0e"    # autumn brown-orange
}

colors = [season_colors[s] for s in season_counts.index]

plt.figure(figsize=(8,4))
ax = season_counts.plot(kind='bar', color=colors, edgecolor='black')

plt.title("24±2h Drift Pairs")
plt.ylabel("Number of Drift Pairs")
plt.xticks(rotation=0)
plt.tight_layout()

# Add numeric labels
for i, value in enumerate(season_counts):
    ax.text(
        i,
        value - max(season_counts) * 0.1,
        str(value),
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

plt.show()


In [ ]:
pairs_df

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import pandas as pd

def plot_drift_pair_starts(pairs_df, bboxes, bbox_colors, title="Drift Pair Start Locations"):
    plt.figure(figsize=(12, 12))
    
    ax = plt.axes(projection=ccrs.NorthPolarStereo())
    ax.set_extent([22, 52, 82, 83], crs=ccrs.PlateCarree())
    ax.set_facecolor("#b0d4f1")
    
    ax.coastlines(color="black", linewidth=1)

    # --- Plot bounding boxes ---
    for name, box in bboxes.items():
        lon1, lat1, lon2, lat2 = box
        color = bbox_colors[name]
        xs = [lon1, lon2, lon2, lon1, lon1]
        ys = [lat1, lat1, lat2, lat2, lat1]
        ax.plot(xs, ys, transform=ccrs.PlateCarree(), color=color, linewidth=3)

    # --- Extract numeric buoy coordinates ---
    df = pairs_df.copy()
    df["t0_lat"] = pd.to_numeric(df.get("t0_lat", df.get("latitude")), errors="coerce")
    df["t0_lon"] = pd.to_numeric(df.get("t0_lon", df.get("longitude")), errors="coerce")

    ax.scatter(
        df["t0_lon"],
        df["t0_lat"],
        s=12,
        color="red",
        transform=ccrs.PlateCarree(),
        label="Drift pair start positions"
    )
    
    ax.gridlines(draw_labels=True)
    plt.title(f"{len(df)} {title}", fontsize=14)
    plt.legend()
    plt.show()


In [ ]:
plot_drift_pair_starts(pairs_df, bboxes, bbox_colors, title="All Drift Pair Start Locations")


In [ ]:
plot_drift_pair_starts(valid_pairs, bboxes, bbox_colors, title="Valid 24±2h Drift Pair Start Locations")


# Looking into a SAR-Buoy drift comparisson

In [ ]:
valid_pairs.head(10)

In [ ]:
import sys
from pathlib import Path

# path to your repo root (where `src` lives)
project_root = Path("..").resolve()  # repo root (buoy_dataset/ is one level down)

# add it to sys.path if not already there
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(sys.path)

In [ ]:
from src.sea_ice_drift.adapted import SeaIceDriftFromTiff, warp_image_with_flow
from datetime import datetime

In [ ]:
valid_pairs.reset_index()[valid_pairs.reset_index()['tiff0_path'].str.contains('region-38_65-82_33-45_61-83_293/2016/11/12/20161112T0557.tiff'
)]

In [ ]:
idx = 369
valid_pairs.reset_index().iloc[idx]

In [ ]:
tiff0_path = valid_pairs.iloc[idx]["tiff0_path"]
tiff1_path = valid_pairs.iloc[idx]["tiff1_path"]
print(tiff0_path)
print(tiff1_path)


In [ ]:
file1 = Path(tiff0_path)
file2 = Path(tiff1_path)

date_time1 = str(file1).split("/")[-1].split(".")[0]
t1 = datetime.strptime(date_time1, "%Y%m%dT%H%M")
date_time2 = str(file2).split("/")[-1].split(".")[0]
t2 = datetime.strptime(date_time2, "%Y%m%dT%H%M")



sid = SeaIceDriftFromTiff(file1, file2, time1=t1, time2=t2, pixel_size_m=100.0)

# --- Feature Tracking preview ---
c1, r1, du_ft, dv_ft = sid.get_drift_FT()
img1 = sid.n1[1]
img2 = sid.n2[1]

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img1, cmap='gray')
plt.title(f"SAR HV - {t1}")
plt.grid(color='r')

plt.subplot(1, 2, 2)
plt.imshow(img2, cmap='gray')
plt.title(f"SAR HV - {t2}")
plt.grid(color='r')

plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 8))
plt.imshow(img1, cmap='gray')
plt.quiver(
    c1, r1, du_ft, dv_ft,
    color='yellow',
    angles='xy', scale_units='xy', scale=1, width=0.003
)
plt.title("FT vectors (pixels)")
# plt.gca().invert_yaxis()
plt.show()

# --- Pattern Matching (PM) ---
u_pix, v_pix, a_deg, r_mcc, h_hess, pm_cols, pm_rows = sid.get_drift_PM(
    grid_step_pix=25,
    img_size=25,
    min_border=40,
    max_border=80,
    threads=4
)

# Basic MCC filter (rMIN = 0.3)
# good = r_mcc >= 0.3 # what is the best threshold here ??
good = (r_mcc * h_hess) > 4

u_f = np.where(good, u_pix, np.nan)
v_f = np.where(good, v_pix, np.nan)

if np.sum(good) == 0:
    raise ValueError("No valid PM vectors to interpolate. Quality mask removed all points.")


u_dense, v_dense = sid.interpolate_to_dense_image_grid(
    pm_cols, pm_rows,
    u_pix, v_pix,
    good_mask=good,
    img_shape=img1.shape,   # e.g. 1024×1024
    method='linear',
    fill_method='nearest'
)

# Downsample dense field for plotting
step = 25  # adjust for clarity (e.g., 10, 20, 40)
rows, cols = img1.shape

rr, cc = np.meshgrid(np.arange(rows), np.arange(cols), indexing='ij')

rr_s = rr[::step, ::step]
cc_s = cc[::step, ::step]

u_s = u_dense[::step, ::step]
v_s = v_dense[::step, ::step]

plt.figure(figsize=(8, 8))
plt.imshow(img1, cmap='gray')
plt.quiver(
    pm_cols, pm_rows, u_f, v_f,
    color='red',
    angles='xy', scale_units='xy', scale=1, width=0.003
)
plt.title("PM vectors (pixels, (r_mcc * h_hess) > 4)")
# plt.gca().invert_yaxis()
plt.show()

plt.figure(figsize=(8,8))
plt.imshow(img1, cmap='gray')
plt.quiver(
    cc_s, rr_s,
    u_s, v_s,
    color='cyan',
    angles='xy', scale_units='xy', scale=1, width=0.003
)
plt.title("Dense interpolated flow field (downsampled)")
plt.show()


In [ ]:
import rioxarray as rxr

def lonlat_to_pixel(da, lon, lat):
    """
    Convert lon/lat → pixel coordinates for a geocoded SAR TIFF.
    da is the rioxarray DataArray (4 bands, y,x).
    """
    # find nearest x-index (lon)
    px = int(np.argmin(np.abs(da.x.values - lon)))
    
    # find nearest y-index (lat)
    py = int(np.argmin(np.abs(da.y.values - lat)))
    
    return px, py


In [ ]:
# Load first TIFF with rioxarray to get geolocation
da1 = rxr.open_rasterio(file1)
da2 = rxr.open_rasterio(file2)

# Get buoy positions
t0_lat = valid_pairs.iloc[idx]["t0_lat"]
t0_lon = valid_pairs.iloc[idx]["t0_lon"]
t1_lat = valid_pairs.iloc[idx]["t1_lat"]
t1_lon = valid_pairs.iloc[idx]["t1_lon"]

# Convert buoy lon/lat → pixel coordinates
bx0, by0 = lonlat_to_pixel(da1, t0_lon, t0_lat)
bx1, by1 = lonlat_to_pixel(da2, t1_lon, t1_lat)


In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img1, cmap="gray")
plt.scatter(bx0, by0, s=50, c="yellow", marker="o", label="Buoy t0")
plt.title(f"SAR HV - {t1}")
plt.grid(color='r')
plt.legend()

plt.subplot(1, 2, 2)
plt.imshow(img2, cmap="gray")
plt.scatter(bx1, by1, s=50, c="yellow", marker="o", label="Buoy t1")
plt.title(f"SAR HV - {t2}")
plt.grid(color='r')
plt.legend()

plt.show()



In [ ]:
# Buoy drift in pixel units
buoy_dx = bx1 - bx0
buoy_dy = by1 - by0

plt.figure(figsize=(8,8))
plt.imshow(img1, cmap='gray')

# dense PM field in cyan
plt.quiver(
    cc_s, rr_s, u_s, v_s,
    color="cyan",
    angles="xy", scale_units="xy", scale=1, width=0.003
)

# buoy drift vector in RED
plt.quiver(
    bx0, by0, buoy_dx, buoy_dy,
    color="red",
    angles="xy", scale_units="xy", scale=1, width=0.005
)

plt.scatter([bx0, bx1], [by0, by1], c=["red", "yellow"], s=50)
plt.title("Dense flow + Buoy drift (red)")
plt.show()


In [ ]:
def compute_buoy_sar_error_meters(
    u_dense, v_dense, 
    bx0, by0, bx1, by1,
    dt_hours,
    pixel_size_m=100.0
):
    """
    Compare buoy drift vector to SAR dense motion vector,
    converting everything into METERS, including endpoint drift error.
    """

    # --- BUOY DRIFT (pixels) ---
    buoy_dx_pix = bx1 - bx0
    buoy_dy_pix = by1 - by0

    # --- BUOY DRIFT (meters) ---
    buoy_dx_m = buoy_dx_pix * pixel_size_m
    buoy_dy_m = buoy_dy_pix * pixel_size_m
    mag_buoy_m = np.hypot(buoy_dx_m, buoy_dy_m)

    # --- SAR DRIFT at buoy start pixel ---
    sar_dx_pix = u_dense[int(by0), int(bx0)]
    sar_dy_pix = v_dense[int(by0), int(bx0)]

    sar_dx_m = sar_dx_pix * pixel_size_m
    sar_dy_m = sar_dy_pix * pixel_size_m
    mag_sar_m = np.hypot(sar_dx_m, sar_dy_m)

    # --- Magnitude error (meters) ---
    mag_error_m = mag_sar_m - mag_buoy_m
    abs_mag_error_m = abs(mag_error_m)

    # --- Direction (angle) error ---
    def angle(dx, dy):
        return np.degrees(np.arctan2(dy, dx))

    angle_buoy = angle(buoy_dx_m, buoy_dy_m)
    angle_sar  = angle(sar_dx_m, sar_dy_m)

    angle_error = (angle_sar - angle_buoy + 180) % 360 - 180

    # --- RMSE (meters) ---
    rmse_pix = np.sqrt((sar_dx_pix - buoy_dx_pix)**2 +
                       (sar_dy_pix - buoy_dy_pix)**2)
    rmse_m = rmse_pix * pixel_size_m

    # --- ENDPOINT ERROR ---
    xf = bx0 + sar_dx_pix   # SAR-predicted final x pixel
    yf = by0 + sar_dy_pix   # SAR-predicted final y pixel

    xb = bx1                # buoy final
    yb = by1

    endpoint_err_pix = np.hypot(xf - xb, yf - yb)
    endpoint_err_m   = endpoint_err_pix * pixel_size_m

    # --- Relative endpoint error ---
    if mag_buoy_m > 0:
        rel_endpoint_err = endpoint_err_m / mag_buoy_m
    else:
        rel_endpoint_err = np.nan

    # --- Speeds (m/s) ---
    dt_seconds = dt_hours * 3600.0
    buoy_speed_ms = mag_buoy_m / dt_seconds
    sar_speed_ms  = mag_sar_m / dt_seconds
    speed_error_ms = sar_speed_ms - buoy_speed_ms

    return {
        # Pixel drift components
        "buoy_dx_pix": buoy_dx_pix,
        "buoy_dy_pix": buoy_dy_pix,
        "sar_dx_pix": sar_dx_pix,
        "sar_dy_pix": sar_dy_pix,

        # Drift vectors in meters
        "buoy_dx_m": buoy_dx_m,
        "buoy_dy_m": buoy_dy_m,
        "sar_dx_m": sar_dx_m,
        "sar_dy_m": sar_dy_m,

        # Magnitudes
        "mag_buoy_m": mag_buoy_m,
        "mag_sar_m": mag_sar_m,
        "mag_error_m": mag_error_m,
        "abs_mag_error_m": abs_mag_error_m,

        # Angular difference
        "angle_buoy_deg": angle_buoy,
        "angle_sar_deg": angle_sar,
        "angle_error_deg": angle_error,

        # RMSE of vector difference
        "rmse_pix": rmse_pix,
        "rmse_m": rmse_m,

        # ENDPOINT ERROR
        "endpoint_err_pix": endpoint_err_pix,
        "endpoint_err_m": endpoint_err_m,
        "rel_endpoint_err": rel_endpoint_err,

        # Speeds
        "buoy_speed_ms": buoy_speed_ms,
        "sar_speed_ms": sar_speed_ms,
        "speed_error_ms": speed_error_ms
    }


In [ ]:
err = compute_buoy_sar_error_meters(
    u_dense, v_dense,
    bx0, by0,
    bx1, by1,
    dt_hours=valid_pairs.iloc[idx]["dt_hours"]
)
for i in ["abs_mag_error_m", "angle_error_deg", "endpoint_err_m", "rel_endpoint_err"]:
    print(f"{i}: {err[i]}")


In [ ]:
# Warp img1 using dense flow
img1_warp = warp_image_with_flow(img1, u_dense, v_dense)

plt.figure(figsize=(18, 6))

# Panel 1 — Original img1
plt.subplot(1, 3, 1)
plt.title("Image 1 (original)")
plt.imshow(img1, cmap='gray_r')
plt.grid(color='tab:red')

# Panel 2 — img1 warped by dense flow
plt.subplot(1, 3, 2)
plt.title("Image 1 warped by dense flow field")
plt.imshow(img1_warp, cmap='gray_r')
plt.grid(color='tab:red')

# Panel 3 — img2
plt.subplot(1, 3, 3)
plt.title("Image 2 (reference)")
plt.imshow(img2, cmap='gray_r')
plt.grid(color='tab:red')

plt.tight_layout()
plt.show()

# Compute all validation stats

In [ ]:
reset_idx_valid_pairs = valid_pairs.reset_index(drop=True)

In [ ]:
# df = reset_idx_valid_pairs.copy()

# # Add output columns
# df["abs_mag_error_m"] = np.nan
# df["angle_error_deg"] = np.nan
# df["endpoint_err_m"] = np.nan
# df["rel_endpoint_err"] = np.nan

# for idx in df.index:

#     print(f"\n=== Processing index {idx} ===")

#     try:
#         # Paths and timestamps
#         file1 = Path(df.loc[idx, "tiff0_path"])
#         file2 = Path(df.loc[idx, "tiff1_path"])

#         t1 = datetime.strptime(file1.stem, "%Y%m%dT%H%M")
#         t2 = datetime.strptime(file2.stem, "%Y%m%dT%H%M")

#         # Sea ice drift object
#         sid = SeaIceDriftFromTiff(
#             file1, file2,
#             time1=t1, time2=t2,
#             pixel_size_m=100.0
#         )

#         # Feature Tracking
#         c1, r1, du_ft, dv_ft = sid.get_drift_FT()
#         img1 = sid.n1[1]  # Needed for dense interpolation target shape

#         # Pattern Matching
#         u_pix, v_pix, a_deg, r_mcc, h_hess, pm_cols, pm_rows = sid.get_drift_PM(
#             grid_step_pix=40,
#             img_size=40,
#             min_border=40,
#             max_border=80,
#             threads=4
#         )

#         # Quality filter
#         good = (r_mcc * h_hess) > 4

#         if np.sum(good) == 0:
#             print(f"[WARNING] No valid PM points for idx={idx}, skipping.")
#             continue  # go to next index

#         # Dense interpolation to full grid
#         u_dense, v_dense = sid.interpolate_to_dense_image_grid(
#             pm_cols, pm_rows,
#             u_pix, v_pix,
#             good_mask=good,
#             img_shape=img1.shape,
#             method='linear',
#             fill_method='nearest'
#         )

#         # Buoy position in pixel coordinates
#         da1 = rxr.open_rasterio(file1)
#         da2 = rxr.open_rasterio(file2)

#         bx0, by0 = lonlat_to_pixel(da1, df.loc[idx, "t0_lon"], df.loc[idx, "t0_lat"])
#         bx1, by1 = lonlat_to_pixel(da2, df.loc[idx, "t1_lon"], df.loc[idx, "t1_lat"])

#         # Compute error between SAR drift and buoy drift
#         err = compute_buoy_sar_error_meters(
#             u_dense, v_dense,
#             bx0, by0,
#             bx1, by1,
#             dt_hours=df.loc[idx, "dt_hours"]
#         )

#         # Store results
#         df.loc[idx, "abs_mag_error_m"] = err["abs_mag_error_m"]
#         df.loc[idx, "angle_error_deg"] = err["angle_error_deg"]
#         df.loc[idx, "endpoint_err_m"] = err["endpoint_err_m"]
#         df.loc[idx, "rel_endpoint_err"] = err["rel_endpoint_err"]

#         print(f"[OK] idx={idx}: endpoint error = {err['endpoint_err_m']:.1f} m, relative endpoint error = {err['rel_endpoint_err']:.2f}")

#     except Exception as e:
#         # If anything crashes, we skip gracefully
#         print(f"[ERROR] idx={idx} failed with: {e}")
#         print("Continuing to next index...")
#         continue

# # Return the updated DF
# df.to_csv("valid_HV_24h_tol2_drift_pairs_with_errors.csv", index=False)
# df.head()


In [ ]:
df_HV = pd.read_csv("valid_HV_24h_tol2_drift_pairs_with_errors.csv")
notna_val = df_HV["endpoint_err_m"].notna().sum()
total_val = len(df_HV)
print(f"Computed endpoint errors for {notna_val} out of {total_val} valid 24±2h drift pairs.")

In [ ]:
def extract_sar_time(tiff_path):
    """Extract SAR timestamp from filename."""
    fname = Path(tiff_path).name.split(".")[0]
    return datetime.strptime(fname, "%Y%m%dT%H%M")


rows = []

for _, row in df_HV.iterrows():
    for label in ["t0", "t1"]:

        tiff_path = row[f"tiff{label[-1]}_path"]

        # --- FIX: convert buoy timestamp from string → datetime ---
        buoy_time = pd.to_datetime(row[label])

        sar_time = extract_sar_time(tiff_path)

        dt_hours = abs((sar_time - buoy_time).total_seconds()) / 3600.0

        rows.append({
            "tiff_path": tiff_path,
            "region": row["region"],
            "buoy_id": row["buoy_id"],
            "buoy_time": buoy_time,
            "sar_time": sar_time,
            "dt_hours": dt_hours
        })

tiff_time_df = pd.DataFrame(rows)
tiff_time_df.head()


In [ ]:
tiff_time_df['dt_hours'].describe()

In [ ]:
tiff_time_df['dt_hours'].hist(bins=50)

In [ ]:
tiff_time_df['dt_hours'][tiff_time_df['dt_hours']<0.5]

In [ ]:
df_HV["rel_endpoint_err"].describe()

In [ ]:
df_HV["endpoint_err_m"].describe()

In [ ]:
# For consistent style
sns.set(style="whitegrid", context="talk")

# Endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HV["endpoint_err_m"],
    color="skyblue",
    inner=None   # remove inner bars (we will add boxplot)
)
sns.boxplot(
    y=df_HV["endpoint_err_m"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True,  # keep outliers for now
)
plt.title("Distribution of Endpoint Error (m)")
plt.ylabel("Endpoint error (m)")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()


# Relative endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HV["rel_endpoint_err"],
    color="lightgreen",
    inner=None
)
sns.boxplot(
    y=df_HV["rel_endpoint_err"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True
)
plt.title("Distribution of Relative Endpoint Error")
plt.ylabel("Relative endpoint error")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
ee_outliers = df_HV[df_HV["endpoint_err_m"] > 5000]  # 5 km threshold
print(len(ee_outliers), "outliers with endpoint error > 5000 m")

rel_ee_outliers = df_HV[df_HV["rel_endpoint_err"] > 10]  # 10 threshold
print(len(rel_ee_outliers), "outliers with relative endpoint error > 10")

In [ ]:
# Use the drift start time t0 (or change to your datetime column)
df_HV["t0"] = pd.to_datetime(df_HV["t0"])

df_HV["month"] = df_HV["t0"].dt.month

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

df_HV["season"] = df_HV["month"].map(season_map)

df_HV_ok = df_HV[df_HV["rel_endpoint_err"] <= 10].dropna(subset=["rel_endpoint_err"])

In [ ]:
season_stats = df_HV_ok.groupby("season")["rel_endpoint_err"].describe()
print("Relative Endpoint Error Statistics by Season:")
season_stats

In [ ]:
season_stats = df_HV_ok.groupby("season")["endpoint_err_m"].describe()
print("Endpoint error stats by season:")
season_stats

In [ ]:
season_stats = df_HV_ok.groupby("season")["angle_error_deg"].describe()
print("Angle error stats by season:")
season_stats

# HH only

In [ ]:
from src.sea_ice_drift.adapted_dualPol import SeaIceDriftFromTiff, warp_image_with_flow

In [ ]:
# df = reset_idx_valid_pairs.copy()

# # Add output columns
# df["abs_mag_error_m"] = np.nan
# df["angle_error_deg"] = np.nan
# df["endpoint_err_m"] = np.nan
# df["rel_endpoint_err"] = np.nan

# for idx in df.index:

#     print(f"\n=== Processing index {idx} ===")

#     try:
#         # Paths and timestamps
#         file1 = Path(df.loc[idx, "tiff0_path"])
#         file2 = Path(df.loc[idx, "tiff1_path"])

#         t1 = datetime.strptime(file1.stem, "%Y%m%dT%H%M")
#         t2 = datetime.strptime(file2.stem, "%Y%m%dT%H%M")

#         # Sea ice drift object
#         sid = SeaIceDriftFromTiff(
#             file1, file2,
#             time1=t1, time2=t2,
#             pixel_size_m=100.0,
#             pol_mode="HH"
#         )

#         # Feature Tracking
#         c1, r1, du_ft, dv_ft = sid.get_drift_FT()
#         img1 = sid.n1[1]  # Needed for dense interpolation target shape

#         # Pattern Matching
#         u_pix, v_pix, a_deg, r_mcc, h_hess, pm_cols, pm_rows = sid.get_drift_PM(
#             grid_step_pix=40,
#             img_size=40,
#             min_border=40,
#             max_border=80,
#             threads=4
#         )

#         # Quality filter
#         good = (r_mcc * h_hess) > 4

#         if np.sum(good) == 0:
#             print(f"[WARNING] No valid PM points for idx={idx}, skipping.")
#             continue  # go to next index

#         # Dense interpolation to full grid
#         u_dense, v_dense = sid.interpolate_to_dense_image_grid(
#             pm_cols, pm_rows,
#             u_pix, v_pix,
#             good_mask=good,
#             img_shape=img1.shape,
#             method='linear',
#             fill_method='nearest'
#         )

#         # Buoy position in pixel coordinates
#         da1 = rxr.open_rasterio(file1)
#         da2 = rxr.open_rasterio(file2)

#         bx0, by0 = lonlat_to_pixel(da1, df.loc[idx, "t0_lon"], df.loc[idx, "t0_lat"])
#         bx1, by1 = lonlat_to_pixel(da2, df.loc[idx, "t1_lon"], df.loc[idx, "t1_lat"])

#         # Compute error between SAR drift and buoy drift
#         err = compute_buoy_sar_error_meters(
#             u_dense, v_dense,
#             bx0, by0,
#             bx1, by1,
#             dt_hours=df.loc[idx, "dt_hours"]
#         )

#         # Store results
#         df.loc[idx, "abs_mag_error_m"] = err["abs_mag_error_m"]
#         df.loc[idx, "angle_error_deg"] = err["angle_error_deg"]
#         df.loc[idx, "endpoint_err_m"] = err["endpoint_err_m"]
#         df.loc[idx, "rel_endpoint_err"] = err["rel_endpoint_err"]

#         print(f"[OK] idx={idx}: endpoint error = {err['endpoint_err_m']:.1f} m, relative endpoint error = {err['rel_endpoint_err']:.2f}")

#     except Exception as e:
#         # If anything crashes, we skip gracefully
#         print(f"[ERROR] idx={idx} failed with: {e}")
#         print("Continuing to next index...")
#         continue

# # Return the updated DF
# df.to_csv("valid_HH_24h_tol2_drift_pairs_with_errors.csv", index=False)
# df.head()

In [ ]:
df_HH = pd.read_csv("valid_HH_24h_tol2_drift_pairs_with_errors.csv")
notna_val = df_HH["rel_endpoint_err"].notna().sum()
total_val = len(df_HH)
print(f"Computed relative endpoint errors for {notna_val} out of {total_val} valid 24±2h drift pairs.")

In [ ]:
df_HH["endpoint_err_m"].describe()

In [ ]:
df_HH["rel_endpoint_err"].describe()

In [ ]:
# For consistent style
sns.set(style="whitegrid", context="talk")

# Endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HH["endpoint_err_m"],
    color="skyblue",
    inner=None   # remove inner bars (we will add boxplot)
)
sns.boxplot(
    y=df_HH["endpoint_err_m"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True,  # keep outliers for now
)
plt.title("Distribution of Endpoint Error (m)")
plt.ylabel("Endpoint error (m)")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()


# Relative endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HH["rel_endpoint_err"],
    color="lightgreen",
    inner=None
)
sns.boxplot(
    y=df_HH["rel_endpoint_err"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True
)
plt.title("Distribution of Relative Endpoint Error")
plt.ylabel("Relative endpoint error")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
ee_outliers = df_HH[df_HH["endpoint_err_m"] > 5000]  # 5 km threshold
print(len(ee_outliers), "outliers with endpoint error > 5000 m")

rel_ee_outliers = df_HH[df_HH["rel_endpoint_err"] > 10]  # 10 threshold
print(len(rel_ee_outliers), "outliers with relative endpoint error > 10")

In [ ]:
# Use the drift start time t0 (or change to your datetime column)
df_HH["t0"] = pd.to_datetime(df_HH["t0"])

df_HH["month"] = df_HH["t0"].dt.month

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

df_HH["season"] = df_HH["month"].map(season_map)

df_HH_ok = df_HH[df_HH["rel_endpoint_err"] <= 10]

In [ ]:
season_stats = df_HH_ok.groupby("season")["rel_endpoint_err"].describe()
print("Relative Endpoint Error Statistics by Season:")
season_stats

In [ ]:
season_stats = df_HH_ok.groupby("season")["endpoint_err_m"].describe()
print("Endpoint error stats by season:")
season_stats

In [ ]:
season_stats = df_HH_ok.groupby("season")["angle_error_deg"].describe()
print("Angle error stats by season:")
season_stats

# HV + HH

In [ ]:
# df = reset_idx_valid_pairs.copy()

# # Add output columns
# df["abs_mag_error_m"] = np.nan
# df["angle_error_deg"] = np.nan
# df["endpoint_err_m"] = np.nan
# df["rel_endpoint_err"] = np.nan

# for idx in df.index:

#     print(f"\n=== Processing index {idx} ===")

#     try:
#         # Paths and timestamps
#         file1 = Path(df.loc[idx, "tiff0_path"])
#         file2 = Path(df.loc[idx, "tiff1_path"])

#         t1 = datetime.strptime(file1.stem, "%Y%m%dT%H%M")
#         t2 = datetime.strptime(file2.stem, "%Y%m%dT%H%M")

#         # Sea ice drift object
#         sid = SeaIceDriftFromTiff(
#             file1, file2,
#             time1=t1, time2=t2,
#             pixel_size_m=100.0,
#             pol_mode="HV+HH"
#         )

#         # Feature Tracking
#         c1, r1, du_ft, dv_ft = sid.get_drift_FT()
#         img1 = sid.n1[1]  # Needed for dense interpolation target shape

#         # Pattern Matching
#         u_pix, v_pix, a_deg, r_mcc, h_hess, pm_cols, pm_rows = sid.get_drift_PM(
#             grid_step_pix=40,
#             img_size=40,
#             min_border=40,
#             max_border=80,
#             threads=4
#         )

#         # Quality filter
#         good = (r_mcc * h_hess) > 4

#         if np.sum(good) == 0:
#             print(f"[WARNING] No valid PM points for idx={idx}, skipping.")
#             continue  # go to next index

#         # Dense interpolation to full grid
#         u_dense, v_dense = sid.interpolate_to_dense_image_grid(
#             pm_cols, pm_rows,
#             u_pix, v_pix,
#             good_mask=good,
#             img_shape=img1.shape,
#             method='linear',
#             fill_method='nearest'
#         )

#         # Buoy position in pixel coordinates
#         da1 = rxr.open_rasterio(file1)
#         da2 = rxr.open_rasterio(file2)

#         bx0, by0 = lonlat_to_pixel(da1, df.loc[idx, "t0_lon"], df.loc[idx, "t0_lat"])
#         bx1, by1 = lonlat_to_pixel(da2, df.loc[idx, "t1_lon"], df.loc[idx, "t1_lat"])

#         # Compute error between SAR drift and buoy drift
#         err = compute_buoy_sar_error_meters(
#             u_dense, v_dense,
#             bx0, by0,
#             bx1, by1,
#             dt_hours=df.loc[idx, "dt_hours"]
#         )

#         # Store results
#         df.loc[idx, "abs_mag_error_m"] = err["abs_mag_error_m"]
#         df.loc[idx, "angle_error_deg"] = err["angle_error_deg"]
#         df.loc[idx, "endpoint_err_m"] = err["endpoint_err_m"]
#         df.loc[idx, "rel_endpoint_err"] = err["rel_endpoint_err"]

#         print(f"[OK] idx={idx}: endpoint error = {err['endpoint_err_m']:.1f} m, relative endpoint error = {err['rel_endpoint_err']:.2f}")

#     except Exception as e:
#         # If anything crashes, we skip gracefully
#         print(f"[ERROR] idx={idx} failed with: {e}")
#         print("Continuing to next index...")
#         continue

# # Return the updated DF
# df.to_csv("valid_HV_HH_24h_tol2_drift_pairs_with_errors.csv", index=False)
# df.head()

In [ ]:
df_HV_HH = pd.read_csv("valid_HV_HH_24h_tol2_drift_pairs_with_errors.csv")
notna_val = df_HV_HH["rel_endpoint_err"].notna().sum()
total_val = len(df_HV_HH)
print(f"Computed relative endpoint errors for {notna_val} out of {total_val} valid 24±2h drift pairs.")

In [ ]:
df_HV_HH["endpoint_err_m"].describe()

In [ ]:
df_HV_HH["rel_endpoint_err"].describe()

In [ ]:
# For consistent style
sns.set(style="whitegrid", context="talk")

# Endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HV_HH["endpoint_err_m"],
    color="skyblue",
    inner=None   # remove inner bars (we will add boxplot)
)
sns.boxplot(
    y=df_HV_HH["endpoint_err_m"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True,  # keep outliers for now
)
plt.title("Distribution of Endpoint Error (m)")
plt.ylabel("Endpoint error (m)")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()


# Relative endpoint error violin + box
plt.figure(figsize=(10,6))
sns.violinplot(
    y=df_HV_HH["rel_endpoint_err"],
    color="lightgreen",
    inner=None
)
sns.boxplot(
    y=df_HV_HH["rel_endpoint_err"],
    width=0.2,
    boxprops={"facecolor":"white"},
    showfliers=True
)
plt.title("Distribution of Relative Endpoint Error")
plt.ylabel("Relative endpoint error")
plt.xlabel("")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Use the drift start time t0 (or change to your datetime column)
df_HV_HH["t0"] = pd.to_datetime(df_HV_HH["t0"])

df_HV_HH["month"] = df_HV_HH["t0"].dt.month

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn",
}

df_HV_HH["season"] = df_HV_HH["month"].map(season_map)

df_HV_HH_ok = df_HV_HH

In [ ]:
season_stats = df_HV_HH_ok.groupby("season")["rel_endpoint_err"].describe()
print("Relative Endpoint Error Statistics by Season:")
season_stats

In [ ]:
season_stats = df_HV_HH_ok.groupby("season")["endpoint_err_m"].describe()
print("Endpoint error stats by season:")
season_stats

In [ ]:
season_stats = df_HV_HH_ok.groupby("season")["angle_error_deg"].describe()
print("Angle error stats by season:")
season_stats

In [ ]:
def compare_full_pipeline_3modes(idx, df=reset_idx_valid_pairs):
    """
    Runs the full SeaIceDrift pipeline for:
        - HH
        - HV
        - HV+HH
    and compares FT, PM, and PM-filtered vectors.

    Produces a 9-panel figure:
        3 rows (FT, PM, PM-filtered)
        3 columns (HH, HV, HV+HH)
    """

    print(f"\n=== FULL PIPELINE COMPARISON for idx={idx} ===")

    # Load file paths and timestamps
    file1 = Path(df.loc[idx, "tiff0_path"])
    file2 = Path(df.loc[idx, "tiff1_path"])

    t1 = datetime.strptime(file1.stem, "%Y%m%dT%H%M")
    t2 = datetime.strptime(file2.stem, "%Y%m%dT%H%M")

    # Load the first image for background
    da = rxr.open_rasterio(file1)
    HH_img = da.isel(band=0).data
    HV_img = da.isel(band=1).data

    # def normalize(img):
    #     img = img.astype("float32")
    #     img = img - np.nanmin(img)
    #     img = img / (np.nanmax(img) + 1e-6)
    #     return img

    # HH_plot = normalize(HH_img)
    # HV_plot = normalize(HV_img)

    HH_plot = 10 * np.log10(HH_img)
    HV_plot = 10 * np.log10(HV_img)
    

    # For HV+HH mode we simply display HV (or could merge)
    HH_HV_plot = HV_plot

    # Helper to run the full pipeline for a given mode
    def run_pipeline(pol_mode):

        print(f"\n--- Running {pol_mode} mode ---")

        sid = SeaIceDriftFromTiff(
            file1, file2,
            time1=t1, time2=t2,
            pixel_size_m=100.0,
            pol_mode=pol_mode
        )

        # -------- FT --------
        c1, r1, du_ft, dv_ft = sid.get_drift_FT()
        N_FT = len(c1)

        # -------- PM --------
        u_pix, v_pix, a_deg, r_mcc, h_hess, pm_cols, pm_rows = sid.get_drift_PM(
            grid_step_pix=25,
            img_size=25,
            min_border=40,
            max_border=80,
            threads=4
        )
        N_PM = np.sum(~np.isnan(u_pix))

        # -------- PM filtered --------
        good = (r_mcc * h_hess) > 4
        good_u = np.where(good, u_pix, np.nan)
        good_v = np.where(good, v_pix, np.nan)
        N_good = np.sum(good)

        return {
            "FT": (c1, r1, du_ft, dv_ft, N_FT),
            "PM": (pm_cols, pm_rows, u_pix, v_pix, N_PM),
            "GOOD": (pm_cols, pm_rows, good_u, good_v, N_good),
        }

    # Run the three pipelines
    HH = run_pipeline("HH")
    HV = run_pipeline("HV")
    HVHH = run_pipeline("HV+HH")

    # Plotting (9 panels)
    plt.figure(figsize=(20, 14))

    background_images = {
        "HH": HH_plot,
        "HV": HV_plot,
        "HV+HH": HH_HV_plot,
    }

    pipelines = {"HH": HH, "HV": HV, "HV+HH": HVHH}
    stages = ["FT", "PM", "GOOD"]
    stage_titles = {
        "FT": "Feature Tracking",
        "PM": "Pattern Matching",
        "GOOD": "PM Filtered (r_mcc * h_hess > 4)"
    }

    mode_colors = {
        "HH": "orange",
        "HV": "red",
        "HV+HH": "magenta"   # highlight combined mode
    }

    for row, stage in enumerate(stages):
        for col, mode in enumerate(["HH", "HV", "HV+HH"]):

            plt.subplot(3, 3, 1 + row*3 + col)

            plt.imshow(background_images[mode], cmap="gray")

            if stage == "FT":
                c1, r1, du, dv, N = pipelines[mode]["FT"]
                plt.quiver(c1, r1, du, dv,
                           color=mode_colors[mode],
                           angles="xy", scale_units="xy",
                           scale=1, width=0.003)
            else:
                cols_grid, rows_grid, u, v, N = pipelines[mode][stage]
                plt.quiver(cols_grid, rows_grid, u, v,
                           color=mode_colors[mode],
                           angles="xy", scale_units="xy",
                           scale=1, width=0.003)

            plt.title(f"{mode} – {stage_titles[stage]} (N={N})")
            plt.axis("off")

    plt.suptitle(f"FT/PM Comparison for idx {idx}: HH vs HV vs HV+HH", fontsize=18)
    plt.tight_layout()
    plt.show()

    return HH, HV, HVHH


In [ ]:
compare_full_pipeline_3modes(6)

In [ ]:
compare_full_pipeline_3modes(7)

In [ ]:
compare_full_pipeline_3modes(8)
